# daat2vec と MLM の特徴表現を互いに予測させる

## 準備

In [2]:
import h5py
import hydra
import numpy as np
import os
import pandas as pd
import pretrain
import random
import torch
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import ttest_ind
from omegaconf import OmegaConf
from pathlib import Path
from tqdm import tqdm

os.chdir(
    "/workspaces/MLMvsData2vec/"
    # "/workspace/"
)  # 変更したいディレクトリのパスを指定
# 現在の作業ディレクトリを確認
print("Current working directory:", os.getcwd())

# 調査するモデルの選択
frameworks = ["mlm", "mlm_spanmask", "data2vec"]    # 実際に計算されるモデルのフレームワーク
model_timestamps = {
    "mlm": "20260316T030756",
    "mlm_spanmask": "20260810T004348",
    "data2vec": "20260324T045257",
}
model_checkpoints = {
    "mlm": 150000,
    "mlm_spanmask": 150000,
    "data2vec": 150000,
}

framework_colors = {
    "data2vec": "orange",
    "mlm": "blue",
    "mlm_spanmask": "green",
    "rinalmo": "gray",
    "onehot_to_mlm": "cyan",
    "onehot_to_mlm_spanmask": "lime",
    "onehot_to_data2vec": "yellow",
}

# RNAに関する情報
nucleotides = ["A", "C", "G", "U"]
family = [
    "5s",
    "16s",
    "23s",
    "grp1",
    "RNaseP",
    "srp",
    "telomerase",
    "tmRNA",
    "tRNA",
]

# ファミリーごとの色（familyは10個なのでtab10を利用）
family_cmap = matplotlib.colormaps["tab10"].colors
family_color_dict = {fam: family_cmap[i] for i, fam in enumerate(family)}

# outputディレクトリの設定
output_dir = Path("results/notebook/predict_another_repr")
output_dir.mkdir(parents=True, exist_ok=True)

# デバイスの設定
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# シードの設定
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

Current working directory: /workspaces/MLMvsData2vec
Using device: cuda


In [5]:
# 表現学習モデルの準備

models = {f"{framework_name}": None for framework_name in frameworks}

pretrain.utils.setup_config()

for framework_name in frameworks:
    if framework_name == "rinalmo":
        continue
    
    framework_name_path = framework_name
    if framework_name == "mlm_spanmask":
        framework_name_path = "mlm"  # mlm_spanmaskはmlmの設定を使う
    
    model_path = Path(f"results/pretrain_results/{framework_name_path}/{model_timestamps[framework_name]}")
    if not model_path.exists():
        raise FileNotFoundError(f"Pretrain model path {model_path} does not exist.")

    cfg_path = model_path / f"train_config/.hydra/config.yaml"
    cfg = OmegaConf.load(cfg_path)
    
    # 互換パッチ: 古い pretrain 実験では `_target_` が "models.*" になっていることがあるため
    # 現在のパッケージ構成に合わせてフルパスに書き換える
    target = getattr(cfg.framework, "_target_", None)
    if target == "models.data2vecModel":
        cfg.framework._target_ = "pretrain.models.data2vecModel"
    elif target == "models.MLMModel":
        cfg.framework._target_ = "pretrain.models.MLMModel"
    
    model: pretrain.models.BaseModel = hydra.utils.instantiate(
        cfg.framework,
        padding_idx=cfg.dataset.tokens.index("<pad>"),
        num_tokens=len(cfg.dataset.tokens),
        experiment_cfg=cfg.experiment,
        device=device
    )

    ## 事前学習モデルの重みの読み込み
    model._load_state_dict(torch.load(model_path / f"weight_{model_checkpoints[framework_name]}.pth", map_location=device))
    
    models[framework_name] = {
        "model": model,
        "cfg": cfg,
    }

/tmp/ipykernel_1595095/2544010660.py:39: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model._load_state_dict(torch.load(model_path / f"weight_{model_checkpoints[framework_n

## 二つの特徴表現を互いに予測させる

ArchiveIIのkfoldを使って，k分割交差検証する

### 準備

In [6]:
# クロスバリデーションの設定
num_folds = 5  # 実際には5に設定することが多いですが、計算時間の関係で1にしています

# データセットの準備
archiveII_df = pd.read_csv("data/SS_data/ArchiveII.csv")
archiveII_kfold_dfs = [
    {
        "train": pd.read_csv(f"data/SS_data/archiveII_kfold/{fold}/train.csv"),
        "test": pd.read_csv(f"data/SS_data/archiveII_kfold/{fold}/test.csv")
    }
    for fold in range(num_folds)
]

# トレーニングの設定
batch_size = 16
max_epochs = 50
warmup_epochs = 5
max_lr = 1e-4
min_lr = 1e-6
early_stopping_patience = 5

# 使用する特徴表現の抽出層の指定
# "all"の場合は全レイヤの表現を抽出, intの場合はその層の表現を抽出, list[int]の場合はその層の表現を抽出
extract_repr_layers = list(range(13))  # 0-12層の表現を抽出 (MLMもData2vecもEmbedding層1層+Transformer層12層あるため)

# 学習率スケジューラーの定義
class CosineScheduler(torch.optim.lr_scheduler._LRScheduler):
    def __init__(
        self,
        optimizer: torch.optim.Optimizer,
        warmup_steps: int = 10000,
        total_steps: int = 200000,
        max_lr: float = 1e-5,
        min_lr: float = 1e-6,
        last_iter: int = -1,
    ):
        self.total_steps = total_steps
        self.warmup_steps = warmup_steps
        self.max_lr = max_lr
        self.min_lr = min_lr
        super().__init__(optimizer, last_epoch=last_iter)

    def get_lr(self):
        last_iter = self.last_epoch
        if last_iter < self.warmup_steps:
            return [base_lr * last_iter / self.warmup_steps for base_lr in self.base_lrs]
        
        if last_iter > self.total_steps:
            return [base_lr * self.min_lr / self.max_lr for base_lr in self.base_lrs]
        
        decay_ratio = (last_iter - self.warmup_steps) / (self.total_steps - self.warmup_steps)
        coeff = 0.5 * (1.0 + np.cos(np.pi * decay_ratio))
        
        return [self.min_lr + coeff * (base_lr - self.min_lr) for base_lr in self.base_lrs]
    

In [11]:
# データローダーの設定

from pretrain.conf.config import MainConfig

def seq2token(
    sequences: list[str],
    tokens: list[str] = ["A", "C", "G", "U", "N", "<mask>", "<pad>", "<cls>", "<eos>"],
    other_tokens: list[str] = ["B", "D", "F", "I", "H", "K", "M", "S", "R", "W", "V", "Y", "X"],
    use_additional_token: bool = False
) -> list[torch.Tensor]:
    """
    文字列のシーケンスをトークンIDのテンソルに変換する関数
    Args:
        sequences (list[str]): 文字列のシーケンスのリスト
        tokens (list[str]): トークンのリスト
        other_tokens (list[str]): その他Nに変換される塩基のリスト
        use_additional_token (bool): CLS, EOSトークンを使用するかどうか
    Returns:
        list[torch.Tensor]: トークンIDのテンソルのリスト
    """
    mapping = {nt: idx for idx, nt in enumerate(tokens)}
    mapping.update({nt: tokens.index("N") for nt in other_tokens})
    mapping["T"] = mapping["U"]
    
    token_seqs = []
    for seq in sequences:
        token_seq = [mapping.get(nt) for nt in seq]
        if use_additional_token:
            token_seq = [mapping["<cls>"]] + token_seq + [mapping["<eos>"]]
        
        if any(v is None for v in token_seq):
            raise ValueError("Invalid nucleotide found")
        token_seqs.append(torch.tensor(token_seq, dtype=torch.uint8))
        
    return token_seqs

class TestDataset(torch.utils.data.Dataset):
    """
    テスト用データセットクラス
    Args:
        dataset_df (pd.DataFrame): データセットのDataFrame
        tokens (list[str]): トークンのリスト
        other_tokens (list[str]): その他トークンのリスト
        use_additional_token (bool): CLS, EOSトークンを使用するかどうか
    """
    
    def __init__(
        self,
        dataset_df: pd.DataFrame,
        tokens: list[str] = ["A", "C", "G", "U", "N", "<mask>", "<pad>", "<cls>", "<eos>"],
        other_tokens: list[str] = ["B", "D", "F", "I", "H", "K", "M", "S", "R", "W", "V", "Y", "X"],
        use_additional_token: bool = False,
    ):
        sequences = dataset_df["sequence"].tolist()
        
        self.seq_ids = dataset_df["id"].tolist()
        self.token_seqs = seq2token(
            sequences,
            tokens=tokens,
            other_tokens=other_tokens,
            use_additional_token=use_additional_token,
        )
        self.tokens = tokens
        
    def __len__(self):
        return len(self.seq_ids)
    
    def __getitem__(self, idx: int):
        return {
            "seq_id": self.seq_ids[idx],
            "token_seq": self.token_seqs[idx],
            "length": len(self.token_seqs[idx]),
        }
        
    def pad_batch(self, batch: list[dict]) -> dict:
        seq_ids = [b["seq_id"] for b in batch]
        token_seqs = [b["token_seq"] for b in batch]
        lengths = [b["length"] for b in batch]
        
        # バディング用にサイズを取得
        batch_size = len(batch)
        max_length = max(lengths)
        
        # バッチ用のテンソルを初期化
        token_seqs_padded = torch.full((batch_size, max_length), fill_value=self.tokens.index("<pad>"), dtype=torch.long)
        
        # attentionマスクの初期化
        attn_mask = torch.full((batch_size, 1, max_length, max_length), fill_value=-1e6)
        
        # パディング
        for k in range(batch_size):
            token_seqs_padded[k, :lengths[k]] = token_seqs[k]
            attn_mask[k, :, :lengths[k], :lengths[k]] = 0
            
        return {
            "seq_ids": seq_ids,
            "token_seqs": token_seqs_padded,
            "attn_mask": attn_mask,
            "attn_biases": None,
            "lengths": lengths,
        }
    
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def create_dataloader(config: MainConfig, split: str, dataset_dfs: dict[str, pd.DataFrame], use_additional_token: bool | None = None):
    """
    データローダーの作成関数
    Args:
        config (MainConfig): 設定情報
        split (str): データ分割 ("train", or "test")
        dataset_dfs (dict[str, pd.DataFrame]): データセットのDataFrameの辞書
        use_additional_token (bool|None): Noneならconfigの値を使用、それ以外は上書き

    Returns:
        torch.utils.data.DataLoader: データローダー
    """
    
    # データセットの選択
    assert split in ["train", "test"], "split must be 'train', or 'test'"
    
    token_flag = config.experiment.use_additional_token if use_additional_token is None else use_additional_token

    dataset = TestDataset(
        dataset_df=dataset_dfs[split],
        tokens=config.dataset.tokens,
        other_tokens=config.dataset.other_tokens,
        use_additional_token=token_flag,
    )

    g = torch.Generator()
    g.manual_seed(seed)
    
    dataloader = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        worker_init_fn=seed_worker,
        generator=g,
        shuffle=False,
        collate_fn=dataset.pad_batch,
    )
    
    return dataloader

In [8]:
# 線形層の次元の設定
mlm_dim = models["mlm"]["cfg"].model_size.embed_dim
if models["mlm"]["cfg"].experiment.use_additional_token:
    mlm_dim = mlm_dim + 2

data2vec_dim = models["data2vec"]["cfg"].model_size.embed_dim
if models["data2vec"]["cfg"].experiment.use_additional_token:
    data2vec_dim = data2vec_dim + 2

### 最終層の特徴表現のみ用いる場合

#### MLM to data2vec

In [21]:
sub_output_dir = output_dir / "mlm_to_data2vec"
sub_output_dir.mkdir(parents=True, exist_ok=True)

for fold in range(num_folds):
    print(f"Fold {fold}:")

    train_loaders = {
        framework_name: create_dataloader(
            config=models[framework_name]["cfg"],
            split="train",
            dataset_dfs=archiveII_kfold_dfs[fold],
            use_additional_token=False,
        )
        for framework_name in frameworks if framework_name != "rinalmo"
    }
    test_loaders = {
        framework_name: create_dataloader(
            config=models[framework_name]["cfg"],
            split="test",
            dataset_dfs=archiveII_kfold_dfs[fold],
            use_additional_token=False,
        )
        for framework_name in frameworks if framework_name != "rinalmo"
    }

    assert len(train_loaders["mlm"]) == len(train_loaders["data2vec"]), "MLM and Data2Vec train loaders must have the same number of batches"
    assert len(test_loaders["mlm"]) == len(test_loaders["data2vec"]), "MLM and Data2Vec test loaders must have the same number of batches"

    num_batches_per_epoch = len(train_loaders["mlm"])
    total_steps = max_epochs * num_batches_per_epoch

    linear = torch.nn.Linear(mlm_dim, data2vec_dim).to(device)
    optimizer = torch.optim.AdamW(linear.parameters(), lr=max_lr)
    scheduler = CosineScheduler(optimizer, warmup_steps=warmup_epochs * num_batches_per_epoch, total_steps=total_steps, max_lr=max_lr, min_lr=min_lr)
    loss_fn = torch.nn.CosineEmbeddingLoss().to(device)

    # トレーニングループ
    print("Training...")

    val_loss = 0.0
    min_val_loss = float("inf")
    metrics_history = {
        "epoch": [],
        "train_loss": [],
        "val_loss": [],
    }
    best_model_state = None
    epochs_without_improvement = 0

    with tqdm(total=total_steps, desc=f"Fold {fold}") as pbar:
        for epoch in range(max_epochs):
            
            total_loss = 0.0
            linear.train()
            for mlm_batch, data2vec_batch in zip(train_loaders["mlm"], train_loaders["data2vec"]):

                with torch.no_grad():
                    mlm_embeddings = models["mlm"]["model"]._test(mlm_batch)["repr"]
                    data2vec_embeddings = models["data2vec"]["model"]._test(data2vec_batch)["repr"]

                # Debug: print shapes and lengths to diagnose mismatch
                try:
                    mlm_lens = mlm_batch.get("lengths", None)
                    data2vec_lens = data2vec_batch.get("lengths", None)
                except Exception:
                    mlm_lens = None
                    data2vec_lens = None

                # print(f"[DEBUG] train epoch={epoch} mlm_batch_size={len(mlm_batch['seq_ids'])} data2vec_batch_size={len(data2vec_batch['seq_ids'])}")
                # print(f"[DEBUG] mlm lengths={mlm_lens} | data2vec lengths={data2vec_lens}")
                # print(f"[DEBUG] mlm_repr_shape={mlm_embeddings.shape} | data2vec_repr_shape={data2vec_embeddings.shape}")

                projected_mlm_embeddings = linear(mlm_embeddings)

                projected_mlm_embeddings = projected_mlm_embeddings.view(-1, projected_mlm_embeddings.size(-1))
                data2vec_embeddings = data2vec_embeddings.view(-1, data2vec_embeddings.size(-1))

                # print(f"[DEBUG] projected_mlm_flat={projected_mlm_embeddings.shape} | data2vec_flat={data2vec_embeddings.shape}")

                if projected_mlm_embeddings.size(0) != data2vec_embeddings.size(0):
                    print("[ERROR] Flattened token counts differ; aborting this batch for inspection")
                    print(f"mlm seq_ids={mlm_batch['seq_ids']}")
                    print(f"data2vec seq_ids={data2vec_batch['seq_ids']}")
                    raise RuntimeError("Batch flattened sizes mismatch")

                loss = loss_fn(projected_mlm_embeddings, data2vec_embeddings, torch.ones(projected_mlm_embeddings.size(0)).to(device))

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                scheduler.step()

                total_loss += loss.item()

                pbar.set_postfix({"loss": f"{loss.item():.4f}", "val_loss": f"{val_loss:.4f}", "lr": f"{scheduler.get_last_lr()[0]:.2e}"})
                pbar.update(1)
            
            avg_loss = total_loss / len(train_loaders["mlm"])
            
            # バリデーション
            total_val_loss = 0.0
            linear.eval()
            for mlm_batch, data2vec_batch in zip(test_loaders["mlm"], test_loaders["data2vec"]):
                with torch.no_grad():
                    mlm_embeddings = models["mlm"]["model"]._test(mlm_batch)["repr"]
                    data2vec_embeddings = models["data2vec"]["model"]._test(data2vec_batch)["repr"]

                projected_mlm_embeddings = linear(mlm_embeddings)

                projected_mlm_embeddings = projected_mlm_embeddings.view(-1, projected_mlm_embeddings.size(-1))
                data2vec_embeddings = data2vec_embeddings.view(-1, data2vec_embeddings.size(-1))

                val_loss = loss_fn(projected_mlm_embeddings, data2vec_embeddings, torch.ones(projected_mlm_embeddings.size(0)).to(device))
                total_val_loss += val_loss.item()

                pbar.set_postfix({"loss": f"{loss.item():.4f}", "val_loss": f"{val_loss.item():.4f}", "lr": f"{scheduler.get_last_lr()[0]:.2e}"})
            
            avg_val_loss = total_val_loss / len(test_loaders["mlm"])

            if avg_val_loss < min_val_loss:
                min_val_loss = avg_val_loss
                best_model_state = linear.state_dict()
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1
            
            if epochs_without_improvement >= early_stopping_patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
            
            metrics_history["epoch"].append(epoch + 1)
            metrics_history["train_loss"].append(avg_loss)
            metrics_history["val_loss"].append(avg_val_loss)
    
    
    metrics_history_df = pd.DataFrame(metrics_history)
    metrics_history_df.to_csv(sub_output_dir / f"fold_{fold}_metrics_history.csv", index=False)
    print(f"Fold {fold} - Training history saved to {sub_output_dir / f'fold_{fold}_metrics_history.csv'}")
    
    # テストループ
    print("Testing...")
    result_dict = {
        "seq_id": [],
        "cosine_similarity": [],
    }
    linear.eval()
    with tqdm(total=len(test_loaders["mlm"]), desc=f"Testing Fold {fold}") as test_pbar:
        for mlm_batch, data2vec_batch in zip(test_loaders["mlm"], test_loaders["data2vec"]):
            with torch.no_grad():
                mlm_embeddings = models["mlm"]["model"]._test(mlm_batch)["repr"]
                data2vec_embeddings = models["data2vec"]["model"]._test(data2vec_batch)["repr"]

            projected_mlm_embeddings = linear(mlm_embeddings)

            projected_mlm_embeddings = projected_mlm_embeddings.view(-1, projected_mlm_embeddings.size(-1)) # (B, L, E) -> (B*L, E)
            data2vec_embeddings = data2vec_embeddings.view(-1, data2vec_embeddings.size(-1))     # (B, L, E) -> (B*L, E)
            
            test_loss = loss_fn(projected_mlm_embeddings, data2vec_embeddings, torch.ones(projected_mlm_embeddings.size(0)).to(device))
            cosine_sim = torch.nn.functional.cosine_similarity(projected_mlm_embeddings, data2vec_embeddings, dim=-1)   # (B*L,)

            cosine_sims = cosine_sim.view(len(mlm_batch["seq_ids"]), -1).mean(dim=-1).detach().cpu().numpy().tolist() # (B, L) -> (B,)

            # 結果をDataFrameに追加
            result_dict["seq_id"].extend(mlm_batch["seq_ids"])
            result_dict["cosine_similarity"].extend(cosine_sims)

            test_pbar.update(1)

    result_df = pd.DataFrame(result_dict)
    result_df.to_csv(sub_output_dir / f"fold_{fold}_results.csv", index=False)
    print(f"Fold {fold} - Test results saved to {sub_output_dir / f'fold_{fold}_results.csv'}")

Fold 0:
Training...


Fold 0: 100%|██████████| 38650/38650 [1:00:39<00:00, 10.62it/s, loss=0.1346, val_loss=0.1405, lr=1.00e-06] 


Fold 0 - Training history saved to results/notebook/predict_another_repr/mlm_to_data2vec/fold_0_metrics_history.csv
Testing...


Testing Fold 0: 100%|██████████| 194/194 [00:14<00:00, 13.54it/s]


Fold 0 - Test results saved to results/notebook/predict_another_repr/mlm_to_data2vec/fold_0_results.csv
Fold 1:
Training...


...

Fold 3 - Test results saved to results/notebook/predict_another_repr/mlm_to_data2vec/fold_3_results.csv
Fold 4:
Training...


Fold 4: 100%|██████████| 38650/38650 [1:00:45<00:00, 10.60it/s, loss=0.1470, val_loss=0.2135, lr=1.00e-06] 


Fold 4 - Training history saved to results/notebook/predict_another_repr/mlm_to_data2vec/fold_4_metrics_history.csv
Testing...


Testing Fold 4: 100%|██████████| 194/194 [00:14<00:00, 13.33it/s]

Fold 4 - Test results saved to results/notebook/predict_another_repr/mlm_to_data2vec/fold_4_results.csv


#### data2vec to MLM

In [22]:
sub_output_dir = output_dir / "data2vec_to_mlm"
sub_output_dir.mkdir(parents=True, exist_ok=True)

for fold in range(num_folds):
    print(f"Fold {fold}:")

    train_loaders = {
        framework_name: create_dataloader(
            config=models[framework_name]["cfg"],
            split="train",
            dataset_dfs=archiveII_kfold_dfs[fold],
            use_additional_token=False,
        )
        for framework_name in frameworks if framework_name != "rinalmo"
    }
    test_loaders = {
        framework_name: create_dataloader(
            config=models[framework_name]["cfg"],
            split="test",
            dataset_dfs=archiveII_kfold_dfs[fold],
            use_additional_token=False,
        )
        for framework_name in frameworks if framework_name != "rinalmo"
    }

    assert len(train_loaders["mlm"]) == len(train_loaders["data2vec"]), "MLM and Data2Vec train loaders must have the same number of batches"
    assert len(test_loaders["mlm"]) == len(test_loaders["data2vec"]), "MLM and Data2Vec test loaders must have the same number of batches"

    num_batches_per_epoch = len(train_loaders["mlm"])
    total_steps = max_epochs * num_batches_per_epoch

    linear = torch.nn.Linear(mlm_dim, data2vec_dim).to(device)
    optimizer = torch.optim.AdamW(linear.parameters(), lr=max_lr)
    scheduler = CosineScheduler(optimizer, warmup_steps=warmup_epochs * num_batches_per_epoch, total_steps=total_steps, max_lr=max_lr, min_lr=min_lr)
    loss_fn = torch.nn.CosineEmbeddingLoss().to(device)

    # トレーニングループ
    print("Training...")

    val_loss = 0.0
    min_val_loss = float("inf")
    metrics_history = {
        "epoch": [],
        "train_loss": [],
        "val_loss": [],
    }
    best_model_state = None
    epochs_without_improvement = 0

    with tqdm(total=total_steps, desc=f"Fold {fold}") as pbar:
        for epoch in range(max_epochs):

            total_loss = 0.0
            linear.train()
            for mlm_batch, data2vec_batch in zip(train_loaders["mlm"], train_loaders["data2vec"]):

                with torch.no_grad():
                    mlm_embeddings = models["mlm"]["model"]._test(mlm_batch)["repr"]
                    data2vec_embeddings = models["data2vec"]["model"]._test(data2vec_batch)["repr"]

                # Debug: print shapes and lengths to diagnose mismatch
                try:
                    mlm_lens = mlm_batch.get("lengths", None)
                    data2vec_lens = data2vec_batch.get("lengths", None)
                except Exception:
                    mlm_lens = None
                    data2vec_lens = None

                # print(f"[DEBUG] train epoch={epoch} mlm_batch_size={len(mlm_batch['seq_ids'])} data2vec_batch_size={len(data2vec_batch['seq_ids'])}")
                # print(f"[DEBUG] mlm lengths={mlm_lens} | data2vec lengths={data2vec_lens}")
                # print(f"[DEBUG] mlm_repr_shape={mlm_embeddings.shape} | data2vec_repr_shape={data2vec_embeddings.shape}")

                projected_data2vec_embeddings = linear(data2vec_embeddings)

                projected_data2vec_embeddings = projected_data2vec_embeddings.view(-1, projected_data2vec_embeddings.size(-1))
                mlm_embeddings = mlm_embeddings.view(-1, mlm_embeddings.size(-1))

                # print(f"[DEBUG] projected_data2vec_flat={projected_data2vec_embeddings.shape} | mlm_flat={mlm_embeddings.shape}")

                if projected_data2vec_embeddings.size(0) != mlm_embeddings.size(0):
                    print("[ERROR] Flattened token counts differ; aborting this batch for inspection")
                    print(f"mlm seq_ids={mlm_batch['seq_ids']}")
                    print(f"data2vec seq_ids={data2vec_batch['seq_ids']}")
                    raise RuntimeError("Batch flattened sizes mismatch")

                loss = loss_fn(projected_data2vec_embeddings, mlm_embeddings, torch.ones(projected_data2vec_embeddings.size(0)).to(device))

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                scheduler.step()

                total_loss += loss.item()

                pbar.set_postfix({"loss": f"{loss.item():.4f}", "val_loss": f"{val_loss:.4f}", "lr": f"{scheduler.get_last_lr()[0]:.2e}"})
                pbar.update(1)
            
            avg_loss = total_loss / len(train_loaders["mlm"])
            
            # バリデーション
            total_val_loss = 0.0
            linear.eval()
            for mlm_batch, data2vec_batch in zip(test_loaders["mlm"], test_loaders["data2vec"]):
                with torch.no_grad():
                    mlm_embeddings = models["mlm"]["model"]._test(mlm_batch)["repr"]
                    data2vec_embeddings = models["data2vec"]["model"]._test(data2vec_batch)["repr"]

                projected_data2vec_embeddings = linear(data2vec_embeddings)

                projected_data2vec_embeddings = projected_data2vec_embeddings.view(-1, projected_data2vec_embeddings.size(-1))
                mlm_embeddings = mlm_embeddings.view(-1, mlm_embeddings.size(-1))

                val_loss = loss_fn(projected_data2vec_embeddings, mlm_embeddings, torch.ones(projected_data2vec_embeddings.size(0)).to(device))
                total_val_loss += val_loss.item()

                pbar.set_postfix({"loss": f"{loss.item():.4f}", "val_loss": f"{val_loss.item():.4f}", "lr": f"{scheduler.get_last_lr()[0]:.2e}"})
            
            avg_val_loss = total_val_loss / len(test_loaders["mlm"])

            if avg_val_loss < min_val_loss:
                min_val_loss = avg_val_loss
                best_model_state = linear.state_dict()
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1
            
            if epochs_without_improvement >= early_stopping_patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
                
            metrics_history["epoch"].append(epoch + 1)
            metrics_history["train_loss"].append(avg_loss)
            metrics_history["val_loss"].append(avg_val_loss)

    metrics_history_df = pd.DataFrame(metrics_history)
    metrics_history_df.to_csv(sub_output_dir / f"fold_{fold}_metrics_history.csv", index=False)
    print(f"Fold {fold} - Training history saved to {sub_output_dir / f'fold_{fold}_metrics_history.csv'}")

    # テストループ
    print("Testing...")
    result_dict = {
        "seq_id": [],
        "cosine_similarity": [],
    }
    linear.eval()
    with tqdm(total=len(test_loaders["mlm"]), desc=f"Testing Fold {fold}") as test_pbar:
        for mlm_batch, data2vec_batch in zip(test_loaders["mlm"], test_loaders["data2vec"]):
            with torch.no_grad():
                mlm_embeddings = models["mlm"]["model"]._test(mlm_batch)["repr"]
                data2vec_embeddings = models["data2vec"]["model"]._test(data2vec_batch)["repr"]

            projected_data2vec_embeddings = linear(data2vec_embeddings)

            projected_data2vec_embeddings = projected_data2vec_embeddings.view(-1, projected_data2vec_embeddings.size(-1))
            mlm_embeddings = mlm_embeddings.view(-1, mlm_embeddings.size(-1))

            test_loss = loss_fn(projected_data2vec_embeddings, mlm_embeddings, torch.ones(projected_data2vec_embeddings.size(0)).to(device))
            cosine_sim = torch.nn.functional.cosine_similarity(projected_data2vec_embeddings, mlm_embeddings, dim=-1)

            cosine_sims = cosine_sim.view(len(mlm_batch["seq_ids"]), -1).mean(dim=-1).detach().cpu().numpy().tolist()

            # 結果をDataFrameに追加
            result_dict["seq_id"].extend(mlm_batch["seq_ids"])
            result_dict["cosine_similarity"].extend(cosine_sims)

    result_df = pd.DataFrame(result_dict)
    result_df.to_csv(sub_output_dir / f"fold_{fold}_results.csv", index=False)
    print(f"Fold {fold} - Test results saved to {sub_output_dir / f'fold_{fold}_results.csv'}")

Fold 0:
Training...


Fold 0: 100%|██████████| 38650/38650 [1:00:15<00:00, 10.69it/s, loss=0.1029, val_loss=0.0710, lr=1.00e-06] 


Fold 0 - Training history saved to results/notebook/predict_another_repr/data2vec_to_mlm/fold_0_metrics_history.csv
Testing...


Testing Fold 0:   0%|          | 0/194 [00:14<?, ?it/s]


Fold 0 - Test results saved to results/notebook/predict_another_repr/data2vec_to_mlm/fold_0_results.csv
Fold 1:
Training...


...

Fold 3 - Test results saved to results/notebook/predict_another_repr/data2vec_to_mlm/fold_3_results.csv
Fold 4:
Training...


Fold 4: 100%|██████████| 38650/38650 [1:00:44<00:00, 10.61it/s, loss=0.0739, val_loss=0.1344, lr=1.00e-06] 


Fold 4 - Training history saved to results/notebook/predict_another_repr/data2vec_to_mlm/fold_4_metrics_history.csv
Testing...


Testing Fold 4:   0%|          | 0/194 [00:14<?, ?it/s]

Fold 4 - Test results saved to results/notebook/predict_another_repr/data2vec_to_mlm/fold_4_results.csv


### 複数の層から抽出された特徴表現を用いる場合

#### MLM to data2vec

In [9]:
sub_output_dir = output_dir / "mlm_to_data2vec"
sub_output_dir.mkdir(parents=True, exist_ok=True)

for fold in range(num_folds):
    print(f"Fold {fold}:")

    train_loaders = {
        framework_name: create_dataloader(
            config=models[framework_name]["cfg"],
            split="train",
            dataset_dfs=archiveII_kfold_dfs[fold],
            use_additional_token=False,
        )
        for framework_name in frameworks if framework_name != "rinalmo"
    }
    test_loaders = {
        framework_name: create_dataloader(
            config=models[framework_name]["cfg"],
            split="test",
            dataset_dfs=archiveII_kfold_dfs[fold],
            use_additional_token=False,
        )
        for framework_name in frameworks if framework_name != "rinalmo"
    }

    assert len(train_loaders["mlm"]) == len(train_loaders["data2vec"]), "MLM and Data2Vec train loaders must have the same number of batches"
    assert len(test_loaders["mlm"]) == len(test_loaders["data2vec"]), "MLM and Data2Vec test loaders must have the same number of batches"

    num_batches_per_epoch = len(train_loaders["mlm"])
    total_steps = max_epochs * num_batches_per_epoch

    linear_list = [torch.nn.Linear(mlm_dim, data2vec_dim).to(device) for _ in range(len(extract_repr_layers))]
    optimizer_list = [torch.optim.AdamW(linear.parameters(), lr=max_lr) for linear in linear_list]
    scheduler_list = [CosineScheduler(optimizer, warmup_steps=warmup_epochs * num_batches_per_epoch, total_steps=total_steps, max_lr=max_lr, min_lr=min_lr) for optimizer in optimizer_list]
    loss_fn = torch.nn.CosineEmbeddingLoss().to(device)

    for index in range(len(extract_repr_layers)):
        terminal_output_dir = sub_output_dir / f"fold_{fold}" / f"layer_{extract_repr_layers[index]}"
        terminal_output_dir.mkdir(parents=True, exist_ok=True)
        
    # トレーニングループ
    print("Training...")
    
    val_loss_list = [0.0 for _ in range(len(extract_repr_layers))]
    min_val_loss_list = [float("inf") for _ in range(len(extract_repr_layers))]
    metrics_history_list = [{
        "epoch": [],
        "train_loss": [],
        "val_loss": [],
    } for _ in range(len(extract_repr_layers))]
    best_model_state_list = [None for _ in range(len(extract_repr_layers))]
    epochs_without_improvement_list = [0 for _ in range(len(extract_repr_layers))]
    
    is_early_stopped = [False for _ in range(len(extract_repr_layers))]
    
    with tqdm(total=total_steps, desc=f"Fold {fold}") as pbar:
        for epoch in range(max_epochs):
            
            total_loss_list = [0.0 for _ in range(len(extract_repr_layers))]
            for linear in linear_list:
                linear.train()
            
            for mlm_batch, data2vec_batch in zip(train_loaders["mlm"], train_loaders["data2vec"]):

                with torch.no_grad():
                    mlm_embeddings_from_multi_layer = models["mlm"]["model"]._test(mlm_batch, extract_repr_layers)["repr"]   # (len(extract_repr_layers), B, L, E)
                    data2vec_embeddings_from_multi_layer = models["data2vec"]["model"]._test(data2vec_batch, extract_repr_layers)["repr"]    # (len(extract_repr_layers), B, L, E)

                if len(extract_repr_layers) == 1:
                    mlm_embeddings_from_multi_layer = mlm_embeddings_from_multi_layer.squeeze(0)
                    data2vec_embeddings_from_multi_layer = data2vec_embeddings_from_multi_layer.squeeze(0)

                for index in range(len(extract_repr_layers)):
                    if is_early_stopped[index]:
                        continue
                    
                    mlm_embeddings = mlm_embeddings_from_multi_layer[index]
                    data2vec_embeddings = data2vec_embeddings_from_multi_layer[index]
                    linear = linear_list[index]
                    optimizer = optimizer_list[index]
                    scheduler = scheduler_list[index]

                    projected_mlm_embeddings = linear(mlm_embeddings)

                    projected_mlm_embeddings = projected_mlm_embeddings.view(-1, projected_mlm_embeddings.size(-1))
                    data2vec_embeddings = data2vec_embeddings.view(-1, data2vec_embeddings.size(-1))

                    loss = loss_fn(projected_mlm_embeddings, data2vec_embeddings, torch.ones(projected_mlm_embeddings.size(0)).to(device))

                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                    scheduler.step()

                    total_loss_list[index] += loss.item()

                pbar.update(1)
            
            avg_loss_list = [total_loss / len(train_loaders["mlm"]) for total_loss in total_loss_list]
            
            # バリデーション
            total_val_loss_list = [0.0 for _ in range(len(extract_repr_layers))]
            for linear in linear_list:
                linear.eval()
            
            for mlm_batch, data2vec_batch in zip(test_loaders["mlm"], test_loaders["data2vec"]):
                with torch.no_grad():
                    mlm_embeddings_from_multi_layer = models["mlm"]["model"]._test(mlm_batch, extract_repr_layers)["repr"]
                    data2vec_embeddings_from_multi_layer = models["data2vec"]["model"]._test(data2vec_batch, extract_repr_layers)["repr"]

                if len(extract_repr_layers) == 1:
                    mlm_embeddings_from_multi_layer = mlm_embeddings_from_multi_layer.squeeze(0)
                    data2vec_embeddings_from_multi_layer = data2vec_embeddings_from_multi_layer.squeeze(0)

                for index in range(len(extract_repr_layers)):
                    if is_early_stopped[index]:
                        continue
                    
                    mlm_embeddings = mlm_embeddings_from_multi_layer[index]
                    data2vec_embeddings = data2vec_embeddings_from_multi_layer[index]
                    linear = linear_list[index]

                    projected_mlm_embeddings = linear(mlm_embeddings)

                    projected_mlm_embeddings = projected_mlm_embeddings.view(-1, projected_mlm_embeddings.size(-1))
                    data2vec_embeddings = data2vec_embeddings.view(-1, data2vec_embeddings.size(-1))

                    val_loss = loss_fn(projected_mlm_embeddings, data2vec_embeddings, torch.ones(projected_mlm_embeddings.size(0)).to(device))
                    total_val_loss_list[index] += val_loss.item()

            avg_val_loss_list = [total_loss / len(test_loaders["mlm"]) for total_loss in total_val_loss_list]

            # 早期終了の判定とモデルの保存
            for index in range(len(extract_repr_layers)):
                if is_early_stopped[index]:
                    continue

                if avg_val_loss_list[index] < min_val_loss_list[index]:
                    min_val_loss_list[index] = avg_val_loss_list[index]
                    best_model_state_list[index] = linear_list[index].state_dict()
                    epochs_without_improvement_list[index] = 0
                else:
                    epochs_without_improvement_list[index] += 1
            
                if epochs_without_improvement_list[index] >= early_stopping_patience:
                    print(f"Layer {extract_repr_layers[index]} early stopped at epoch {epoch+1}")
                    is_early_stopped[index] = True
                
                metrics_history_list[index]["epoch"].append(epoch + 1)
                metrics_history_list[index]["train_loss"].append(avg_loss_list[index])
                metrics_history_list[index]["val_loss"].append(avg_val_loss_list[index])

            if all(is_early_stopped):
                print(f"All layers early stopped at epoch {epoch+1}")
                break
    
    for index in range(len(extract_repr_layers)):
        metrics_history_df = pd.DataFrame(metrics_history_list[index])
        metrics_history_df.to_csv(sub_output_dir / f"fold_{fold}" / f"layer_{extract_repr_layers[index]}" / "metrics_history.csv", index=False)
        print(f"Fold {fold} - Training history saved to {sub_output_dir / f'fold_{fold}' / f'layer_{extract_repr_layers[index]}' / 'metrics_history.csv'}")
        
        best_model_state = best_model_state_list[index]
        if best_model_state is not None:
            torch.save(best_model_state, sub_output_dir / f"fold_{fold}" / f"layer_{extract_repr_layers[index]}" / "best_model.pth")
            print(f"Fold {fold} - Best model saved to {sub_output_dir / f'fold_{fold}' / f'layer_{extract_repr_layers[index]}' / 'best_model.pth'}")
    
    # テストループ
    print("Testing...")
    
    result_dict_list = [{
        "seq_id": [],
        f"cosine_similarity_{extract_repr_layers[index]}": [],
    } for index in range(len(extract_repr_layers))]
    
    for linear in linear_list:
        linear.eval()
    
    with tqdm(total=len(test_loaders["mlm"]), desc=f"Testing Fold {fold}") as test_pbar:
        for mlm_batch, data2vec_batch in zip(test_loaders["mlm"], test_loaders["data2vec"]):
            with torch.no_grad():
                mlm_embeddings_from_multi_layers = models["mlm"]["model"]._test(mlm_batch, extract_repr_layers)["repr"]
                data2vec_embeddings_from_multi_layers = models["data2vec"]["model"]._test(data2vec_batch, extract_repr_layers)["repr"]

            for index in range(len(extract_repr_layers)):
                mlm_embeddings = mlm_embeddings_from_multi_layers[index]
                data2vec_embeddings = data2vec_embeddings_from_multi_layers[index]
                linear = linear_list[index]
                
                projected_mlm_embeddings = linear(mlm_embeddings)

                projected_mlm_embeddings = projected_mlm_embeddings.view(-1, projected_mlm_embeddings.size(-1)) # (B, L, E) -> (B*L, E)
                data2vec_embeddings = data2vec_embeddings.view(-1, data2vec_embeddings.size(-1))     # (B, L, E) -> (B*L, E)
                
                test_loss = loss_fn(projected_mlm_embeddings, data2vec_embeddings, torch.ones(projected_mlm_embeddings.size(0)).to(device))
                cosine_sim = torch.nn.functional.cosine_similarity(projected_mlm_embeddings, data2vec_embeddings, dim=-1)   # (B*L,)

                cosine_sims = cosine_sim.view(len(mlm_batch["seq_ids"]), -1).mean(dim=-1).detach().cpu().numpy().tolist() # (B, L) -> (B,)

                # 結果をDataFrameに追加
                result_dict_list[index]["seq_id"].extend(mlm_batch["seq_ids"])
                result_dict_list[index][f"cosine_similarity_{extract_repr_layers[index]}"].extend(cosine_sims)

    df_list = []
    for index in range(len(extract_repr_layers)):
        df_list.append(pd.DataFrame(result_dict_list[index]))
    
    result_df = pd.concat(df_list, axis=1)
    result_df.to_csv(sub_output_dir / f"fold_{fold}" / f"results.csv", index=False)
    print(f"Fold {fold} - Test results saved to {sub_output_dir / f'fold_{fold}' / f'results.csv'}")

all_folds_results = []
for fold in range(num_folds):
    fold_result_df = pd.read_csv(sub_output_dir / f"fold_{fold}" / f"results.csv")
    fold_result_df["fold"] = fold
    all_folds_results.append(fold_result_df)
final_result_df = pd.concat(all_folds_results, axis=0)
final_result_df.to_csv(sub_output_dir / f"all_folds_results.csv", index=False)
print(f"All folds results saved to {sub_output_dir / f'all_folds_results.csv'}")

Fold 0:
Training...


Fold 0:  52%|█████▏    | 5043/9700 [42:38<28:41,  2.71it/s]   

Layer 0 early stopped at epoch 26


Fold 0: 100%|██████████| 9700/9700 [1:22:28<00:00,  1.96it/s]  


Fold 0 - Training history saved to results/notebook/predict_another_repr/mlm_to_data2vec/fold_0/layer_0/metrics_history.csv
Fold 0 - Best model saved to results/notebook/predict_another_repr/mlm_to_data2vec/fold_0/layer_0/best_model.pth
Fold 0 - Training history saved to results/notebook/predict_another_repr/mlm_to_data2vec/fold_0/layer_1/metrics_history.csv
Fold 0 - Best model saved to results/notebook/predict_another_repr/mlm_to_data2vec/fold_0/layer_1/best_model.pth
Fold 0 - Training history saved to results/notebook/predict_another_repr/mlm_to_data2vec/fold_0/layer_2/metrics_history.csv
Fold 0 - Best model saved to results/notebook/predict_another_repr/mlm_to_data2vec/fold_0/layer_2/best_model.pth
Fold 0 - Training history saved to results/notebook/predict_another_repr/mlm_to_data2vec/fold_0/layer_3/metrics_history.csv
Fold 0 - Best model saved to results/notebook/predict_another_repr/mlm_to_data2vec/fold_0/layer_3/best_model.pth
Fold 0 - Training history saved to results/notebook/

...

Layer 0 early stopped at epoch 13


Fold 4: 100%|██████████| 9700/9700 [1:23:22<00:00,  1.94it/s]  


Fold 4 - Training history saved to results/notebook/predict_another_repr/mlm_to_data2vec/fold_4/layer_0/metrics_history.csv
Fold 4 - Best model saved to results/notebook/predict_another_repr/mlm_to_data2vec/fold_4/layer_0/best_model.pth
Fold 4 - Training history saved to results/notebook/predict_another_repr/mlm_to_data2vec/fold_4/layer_1/metrics_history.csv
Fold 4 - Best model saved to results/notebook/predict_another_repr/mlm_to_data2vec/fold_4/layer_1/best_model.pth
Fold 4 - Training history saved to results/notebook/predict_another_repr/mlm_to_data2vec/fold_4/layer_2/metrics_history.csv
Fold 4 - Best model saved to results/notebook/predict_another_repr/mlm_to_data2vec/fold_4/layer_2/best_model.pth
Fold 4 - Training history saved to results/notebook/predict_another_repr/mlm_to_data2vec/fold_4/layer_3/metrics_history.csv
Fold 4 - Best model saved to results/notebook/predict_another_repr/mlm_to_data2vec/fold_4/layer_3/best_model.pth
Fold 4 - Training history saved to results/notebook/

Testing Fold 4:   0%|          | 0/49 [00:19<?, ?it/s]

Fold 4 - Test results saved to results/notebook/predict_another_repr/mlm_to_data2vec/fold_4/results.csv
All folds results saved to results/notebook/predict_another_repr/mlm_to_data2vec/all_folds_results.csv


#### data2vec to MLM

In [10]:
sub_output_dir = output_dir / "data2vec_to_mlm"
sub_output_dir.mkdir(parents=True, exist_ok=True)

for fold in range(num_folds):
    print(f"Fold {fold}:")

    train_loaders = {
        framework_name: create_dataloader(
            config=models[framework_name]["cfg"],
            split="train",
            dataset_dfs=archiveII_kfold_dfs[fold],
            use_additional_token=False,
        )
        for framework_name in frameworks if framework_name != "rinalmo"
    }
    test_loaders = {
        framework_name: create_dataloader(
            config=models[framework_name]["cfg"],
            split="test",
            dataset_dfs=archiveII_kfold_dfs[fold],
            use_additional_token=False,
        )
        for framework_name in frameworks if framework_name != "rinalmo"
    }

    assert len(train_loaders["mlm"]) == len(train_loaders["data2vec"]), "MLM and Data2Vec train loaders must have the same number of batches"
    assert len(test_loaders["mlm"]) == len(test_loaders["data2vec"]), "MLM and Data2Vec test loaders must have the same number of batches"

    num_batches_per_epoch = len(train_loaders["data2vec"])
    total_steps = max_epochs * num_batches_per_epoch

    linear_list = [torch.nn.Linear(data2vec_dim, mlm_dim).to(device) for _ in range(len(extract_repr_layers))]
    optimizer_list = [torch.optim.AdamW(linear.parameters(), lr=max_lr) for linear in linear_list]
    scheduler_list = [CosineScheduler(optimizer, warmup_steps=warmup_epochs * num_batches_per_epoch, total_steps=total_steps, max_lr=max_lr, min_lr=min_lr) for optimizer in optimizer_list]
    loss_fn = torch.nn.CosineEmbeddingLoss().to(device)

    for index in range(len(extract_repr_layers)):
        terminal_output_dir = sub_output_dir / f"fold_{fold}" / f"layer_{extract_repr_layers[index]}"
        terminal_output_dir.mkdir(parents=True, exist_ok=True)
        
    # トレーニングループ
    print("Training...")
    
    val_loss_list = [0.0 for _ in range(len(extract_repr_layers))]
    min_val_loss_list = [float("inf") for _ in range(len(extract_repr_layers))]
    metrics_history_list = [{
        "epoch": [],
        "train_loss": [],
        "val_loss": [],
    } for _ in range(len(extract_repr_layers))]
    best_model_state_list = [None for _ in range(len(extract_repr_layers))]
    epochs_without_improvement_list = [0 for _ in range(len(extract_repr_layers))]
    
    is_early_stopped = [False for _ in range(len(extract_repr_layers))]
    
    with tqdm(total=total_steps, desc=f"Fold {fold}") as pbar:
        for epoch in range(max_epochs):
            
            total_loss_list = [0.0 for _ in range(len(extract_repr_layers))]
            for linear in linear_list:
                linear.train()
            
            for mlm_batch, data2vec_batch in zip(train_loaders["mlm"], train_loaders["data2vec"]):

                with torch.no_grad():
                    mlm_embeddings_from_multi_layer = models["mlm"]["model"]._test(mlm_batch, extract_repr_layers)["repr"]   # (len(extract_repr_layers), B, L, E)
                    data2vec_embeddings_from_multi_layer = models["data2vec"]["model"]._test(data2vec_batch, extract_repr_layers)["repr"]    # (len(extract_repr_layers), B, L, E)

                if len(extract_repr_layers) == 1:
                    mlm_embeddings_from_multi_layer = mlm_embeddings_from_multi_layer.squeeze(0)
                    data2vec_embeddings_from_multi_layer = data2vec_embeddings_from_multi_layer.squeeze(0)

                for index in range(len(extract_repr_layers)):
                    if is_early_stopped[index]:
                        continue
                    
                    mlm_embeddings = mlm_embeddings_from_multi_layer[index]
                    data2vec_embeddings = data2vec_embeddings_from_multi_layer[index]
                    linear = linear_list[index]
                    optimizer = optimizer_list[index]
                    scheduler = scheduler_list[index]

                    projected_data2vec_embeddings = linear(data2vec_embeddings)

                    projected_data2vec_embeddings = projected_data2vec_embeddings.view(-1, projected_data2vec_embeddings.size(-1))
                    mlm_embeddings = mlm_embeddings.view(-1, mlm_embeddings.size(-1))

                    loss = loss_fn(projected_data2vec_embeddings, mlm_embeddings, torch.ones(projected_data2vec_embeddings.size(0)).to(device))

                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                    scheduler.step()

                    total_loss_list[index] += loss.item()

                pbar.update(1)
            
            avg_loss_list = [total_loss / len(train_loaders["mlm"]) for total_loss in total_loss_list]
            
            # バリデーション
            total_val_loss_list = [0.0 for _ in range(len(extract_repr_layers))]
            for linear in linear_list:
                linear.eval()
            
            for mlm_batch, data2vec_batch in zip(test_loaders["mlm"], test_loaders["data2vec"]):
                with torch.no_grad():
                    mlm_embeddings_from_multi_layer = models["mlm"]["model"]._test(mlm_batch, extract_repr_layers)["repr"]
                    data2vec_embeddings_from_multi_layer = models["data2vec"]["model"]._test(data2vec_batch, extract_repr_layers)["repr"]

                if len(extract_repr_layers) == 1:
                    mlm_embeddings_from_multi_layer = mlm_embeddings_from_multi_layer.squeeze(0)
                    data2vec_embeddings_from_multi_layer = data2vec_embeddings_from_multi_layer.squeeze(0)

                for index in range(len(extract_repr_layers)):
                    if is_early_stopped[index]:
                        continue
                    
                    mlm_embeddings = mlm_embeddings_from_multi_layer[index]
                    data2vec_embeddings = data2vec_embeddings_from_multi_layer[index]
                    linear = linear_list[index]

                    projected_data2vec_embeddings = linear(data2vec_embeddings)

                    projected_data2vec_embeddings = projected_data2vec_embeddings.view(-1, projected_data2vec_embeddings.size(-1))
                    mlm_embeddings = mlm_embeddings.view(-1, mlm_embeddings.size(-1))

                    val_loss = loss_fn(projected_data2vec_embeddings, mlm_embeddings, torch.ones(projected_data2vec_embeddings.size(0)).to(device))
                    total_val_loss_list[index] += val_loss.item()

            avg_val_loss_list = [total_loss / len(test_loaders["mlm"]) for total_loss in total_val_loss_list]

            # 早期終了の判定とモデルの保存
            for index in range(len(extract_repr_layers)):
                if is_early_stopped[index]:
                    continue

                if avg_val_loss_list[index] < min_val_loss_list[index]:
                    min_val_loss_list[index] = avg_val_loss_list[index]
                    best_model_state_list[index] = linear_list[index].state_dict()
                    epochs_without_improvement_list[index] = 0
                else:
                    epochs_without_improvement_list[index] += 1
            
                if epochs_without_improvement_list[index] >= early_stopping_patience:
                    print(f"Layer {extract_repr_layers[index]} early stopped at epoch {epoch+1}")
                    is_early_stopped[index] = True
                
                metrics_history_list[index]["epoch"].append(epoch + 1)
                metrics_history_list[index]["train_loss"].append(avg_loss_list[index])
                metrics_history_list[index]["val_loss"].append(avg_val_loss_list[index])

            if all(is_early_stopped):
                print(f"All layers early stopped at epoch {epoch+1}")
                break
    
    for index in range(len(extract_repr_layers)):
        metrics_history_df = pd.DataFrame(metrics_history_list[index])
        metrics_history_df.to_csv(sub_output_dir / f"fold_{fold}" / f"layer_{extract_repr_layers[index]}" / "metrics_history.csv", index=False)
        print(f"Fold {fold} - Training history saved to {sub_output_dir / f'fold_{fold}' / f'layer_{extract_repr_layers[index]}' / 'metrics_history.csv'}")
        
        best_model_state = best_model_state_list[index]
        if best_model_state is not None:
            torch.save(best_model_state, sub_output_dir / f"fold_{fold}" / f"layer_{extract_repr_layers[index]}" / "best_model.pth")
            print(f"Fold {fold} - Best model saved to {sub_output_dir / f'fold_{fold}' / f'layer_{extract_repr_layers[index]}' / 'best_model.pth'}")
    
    # テストループ
    print("Testing...")
    
    result_dict_list = [{
        "seq_id": [],
        f"cosine_similarity_{extract_repr_layers[index]}": [],
    } for index in range(len(extract_repr_layers))]
    
    for linear in linear_list:
        linear.eval()
    
    with tqdm(total=len(test_loaders["mlm"]), desc=f"Testing Fold {fold}") as test_pbar:
        for mlm_batch, data2vec_batch in zip(test_loaders["mlm"], test_loaders["data2vec"]):
            with torch.no_grad():
                mlm_embeddings_from_multi_layers = models["mlm"]["model"]._test(mlm_batch, extract_repr_layers)["repr"]
                data2vec_embeddings_from_multi_layers = models["data2vec"]["model"]._test(data2vec_batch, extract_repr_layers)["repr"]

            for index in range(len(extract_repr_layers)):
                mlm_embeddings = mlm_embeddings_from_multi_layers[index]
                data2vec_embeddings = data2vec_embeddings_from_multi_layers[index]
                linear = linear_list[index]
                
                projected_data2vec_embeddings = linear(data2vec_embeddings)

                projected_data2vec_embeddings = projected_data2vec_embeddings.view(-1, projected_data2vec_embeddings.size(-1)) # (B, L, E) -> (B*L, E)
                mlm_embeddings = mlm_embeddings.view(-1, mlm_embeddings.size(-1))     # (B, L, E) -> (B*L, E)
                
                test_loss = loss_fn(projected_data2vec_embeddings, mlm_embeddings, torch.ones(projected_data2vec_embeddings.size(0)).to(device))
                cosine_sim = torch.nn.functional.cosine_similarity(projected_data2vec_embeddings, mlm_embeddings, dim=-1)   # (B*L,)

                cosine_sims = cosine_sim.view(len(mlm_batch["seq_ids"]), -1).mean(dim=-1).detach().cpu().numpy().tolist() # (B, L) -> (B,)

                # 結果をDataFrameに追加
                result_dict_list[index]["seq_id"].extend(mlm_batch["seq_ids"])
                result_dict_list[index][f"cosine_similarity_{extract_repr_layers[index]}"].extend(cosine_sims)

    df_list = []
    for index in range(len(extract_repr_layers)):
        df_list.append(pd.DataFrame(result_dict_list[index]))
    
    result_df = pd.concat(df_list, axis=1)
    result_df.to_csv(sub_output_dir / f"fold_{fold}" / f"results.csv", index=False)
    print(f"Fold {fold} - Test results saved to {sub_output_dir / f'fold_{fold}' / f'results.csv'}")

all_folds_results = []
for fold in range(num_folds):
    fold_result_df = pd.read_csv(sub_output_dir / f"fold_{fold}" / f"results.csv")
    fold_result_df["fold"] = fold
    all_folds_results.append(fold_result_df)
final_result_df = pd.concat(all_folds_results, axis=0)
final_result_df.to_csv(sub_output_dir / f"all_folds_results.csv", index=False)
print(f"All folds results saved to {sub_output_dir / f'all_folds_results.csv'}")

Fold 0:
Training...


Fold 0:  28%|██▊       | 2715/9700 [22:48<43:09,  2.70it/s]   

Layer 0 early stopped at epoch 14


Fold 0: 100%|██████████| 9700/9700 [1:22:23<00:00,  1.96it/s]  


Fold 0 - Training history saved to results/notebook/predict_another_repr/data2vec_to_mlm/fold_0/layer_0/metrics_history.csv
Fold 0 - Best model saved to results/notebook/predict_another_repr/data2vec_to_mlm/fold_0/layer_0/best_model.pth
Fold 0 - Training history saved to results/notebook/predict_another_repr/data2vec_to_mlm/fold_0/layer_1/metrics_history.csv
Fold 0 - Best model saved to results/notebook/predict_another_repr/data2vec_to_mlm/fold_0/layer_1/best_model.pth
Fold 0 - Training history saved to results/notebook/predict_another_repr/data2vec_to_mlm/fold_0/layer_2/metrics_history.csv
Fold 0 - Best model saved to results/notebook/predict_another_repr/data2vec_to_mlm/fold_0/layer_2/best_model.pth
Fold 0 - Training history saved to results/notebook/predict_another_repr/data2vec_to_mlm/fold_0/layer_3/metrics_history.csv
Fold 0 - Best model saved to results/notebook/predict_another_repr/data2vec_to_mlm/fold_0/layer_3/best_model.pth
Fold 0 - Training history saved to results/notebook/

...

Layer 0 early stopped at epoch 18


Fold 4: 100%|██████████| 9700/9700 [1:23:27<00:00,  1.94it/s]  


Fold 4 - Training history saved to results/notebook/predict_another_repr/data2vec_to_mlm/fold_4/layer_0/metrics_history.csv
Fold 4 - Best model saved to results/notebook/predict_another_repr/data2vec_to_mlm/fold_4/layer_0/best_model.pth
Fold 4 - Training history saved to results/notebook/predict_another_repr/data2vec_to_mlm/fold_4/layer_1/metrics_history.csv
Fold 4 - Best model saved to results/notebook/predict_another_repr/data2vec_to_mlm/fold_4/layer_1/best_model.pth
Fold 4 - Training history saved to results/notebook/predict_another_repr/data2vec_to_mlm/fold_4/layer_2/metrics_history.csv
Fold 4 - Best model saved to results/notebook/predict_another_repr/data2vec_to_mlm/fold_4/layer_2/best_model.pth
Fold 4 - Training history saved to results/notebook/predict_another_repr/data2vec_to_mlm/fold_4/layer_3/metrics_history.csv
Fold 4 - Best model saved to results/notebook/predict_another_repr/data2vec_to_mlm/fold_4/layer_3/best_model.pth
Fold 4 - Training history saved to results/notebook/

Testing Fold 4:   0%|          | 0/49 [00:19<?, ?it/s]

Fold 4 - Test results saved to results/notebook/predict_another_repr/data2vec_to_mlm/fold_4/results.csv
All folds results saved to results/notebook/predict_another_repr/data2vec_to_mlm/all_folds_results.csv


#### onehot to mlm / data2vec

In [9]:
# onehotデータの取得
onehot_data = h5py.File("data/embeddings/one-hot_ArchiveII.h5", "r")

In [12]:
dim_map = {
    "mlm": mlm_dim,
    "data2vec": data2vec_dim,
}

for to_framework in ["mlm", "data2vec"]:
    sub_output_dir = output_dir / f"onehot_to_{to_framework}"
    sub_output_dir.mkdir(parents=True, exist_ok=True)

    for fold in range(num_folds):
        print(f"Fold {fold}:")

        train_loader = create_dataloader(
                config=models[to_framework]["cfg"],
                split="train",
                dataset_dfs=archiveII_kfold_dfs[fold],
                use_additional_token=False,
            )
        
        test_loader = create_dataloader(
                config=models[to_framework]["cfg"],
                split="test",
                dataset_dfs=archiveII_kfold_dfs[fold],
                use_additional_token=False,
            )

        num_batches_per_epoch = len(train_loader)
        total_steps = max_epochs * num_batches_per_epoch

        linear_list = [torch.nn.Linear(4, dim_map[to_framework]).to(device) for _ in range(len(extract_repr_layers))]
        optimizer_list = [torch.optim.AdamW(linear.parameters(), lr=max_lr) for linear in linear_list]
        scheduler_list = [CosineScheduler(optimizer, warmup_steps=warmup_epochs * num_batches_per_epoch, total_steps=total_steps, max_lr=max_lr, min_lr=min_lr) for optimizer in optimizer_list]
        loss_fn = torch.nn.CosineEmbeddingLoss().to(device)

        for index in range(len(extract_repr_layers)):
            terminal_output_dir = sub_output_dir / f"fold_{fold}" / f"layer_{extract_repr_layers[index]}"
            terminal_output_dir.mkdir(parents=True, exist_ok=True)
            
        # トレーニングループ
        print("Training...")
        
        val_loss_list = [0.0 for _ in range(len(extract_repr_layers))]
        min_val_loss_list = [float("inf") for _ in range(len(extract_repr_layers))]
        metrics_history_list = [{
            "epoch": [],
            "train_loss": [],
            "val_loss": [],
        } for _ in range(len(extract_repr_layers))]
        best_model_state_list = [None for _ in range(len(extract_repr_layers))]
        epochs_without_improvement_list = [0 for _ in range(len(extract_repr_layers))]
        
        is_early_stopped = [False for _ in range(len(extract_repr_layers))]
        
        with tqdm(total=total_steps, desc=f"Fold {fold}") as pbar:
            for epoch in range(max_epochs):
                
                total_loss_list = [0.0 for _ in range(len(extract_repr_layers))]
                for linear in linear_list:
                    linear.train()
                
                for batch in train_loader:
                    seq_ids = batch["seq_ids"]
                    max_length = max(batch["lengths"])
                    
                    # onehotデータの取得とパディング, テンソル化
                    onehot_vectors = []
                    for seq_id in seq_ids:
                        onehot_vector = onehot_data[seq_id][:max_length]  # (L, 4)
                        if onehot_vector.shape[0] < max_length:
                            padding = np.zeros((max_length - onehot_vector.shape[0], 4))
                            onehot_vector = np.vstack([onehot_vector, padding])
                        onehot_vectors.append(onehot_vector)
                    onehot_vectors = np.stack(onehot_vectors, axis=0)  # (B, L, 4)
                    onehot_vectors = torch.tensor(onehot_vectors, dtype=torch.float32).to(device)  # (B, L, 4)

                    with torch.no_grad():
                        targets = models[f"{to_framework}"]["model"]._test(batch, extract_repr_layers)["repr"]   # (len(extract_repr_layers), B, L, E)

                    if len(extract_repr_layers) == 1:
                        targets = targets.squeeze(0)  # (B, L, E)

                    for index in range(len(extract_repr_layers)):
                        if is_early_stopped[index]:
                            continue
                        
                        target = targets[index] # (B, L, E)
                        linear = linear_list[index]
                        optimizer = optimizer_list[index]
                        scheduler = scheduler_list[index]

                        projected_onehot_vectors = linear(onehot_vectors)

                        projected_onehot_vectors = projected_onehot_vectors.view(-1, projected_onehot_vectors.size(-1))
                        target = target.view(-1, target.size(-1))

                        loss = loss_fn(projected_onehot_vectors, target, torch.ones(projected_onehot_vectors.size(0)).to(device))

                        optimizer.zero_grad()
                        loss.backward()
                        optimizer.step()
                        scheduler.step()

                        total_loss_list[index] += loss.item()

                    pbar.update(1)
                
                avg_loss_list = [total_loss / len(train_loader) for total_loss in total_loss_list]
                
                # バリデーション
                total_val_loss_list = [0.0 for _ in range(len(extract_repr_layers))]
                for linear in linear_list:
                    linear.eval()
                
                for batch in test_loader:
                    seq_ids = batch["seq_ids"]
                    max_length = max(batch["lengths"])
                    # onehotデータの取得とパディング, テンソル化
                    onehot_vectors = []
                    for seq_id in seq_ids:
                        onehot_vector = onehot_data[seq_id][:max_length]  # (L, 4)
                        if onehot_vector.shape[0] < max_length:
                            padding = np.zeros((max_length - onehot_vector.shape[0], 4))
                            onehot_vector = np.vstack([onehot_vector, padding])
                        onehot_vectors.append(onehot_vector)
                    onehot_vectors = np.stack(onehot_vectors, axis=0)  # (B, L, 4)
                    onehot_vectors = torch.tensor(onehot_vectors, dtype=torch.float32).to(device)  # (B, L, 4)
                    
                    with torch.no_grad():
                        targets = models[f"{to_framework}"]["model"]._test(batch, extract_repr_layers)["repr"]

                    if len(extract_repr_layers) == 1:
                        targets = targets.squeeze(0)

                    for index in range(len(extract_repr_layers)):
                        if is_early_stopped[index]:
                            continue
                        
                        target = targets[index]
                        linear = linear_list[index]

                        projected_onehot_vectors = linear(onehot_vectors)

                        projected_onehot_vectors = projected_onehot_vectors.view(-1, projected_onehot_vectors.size(-1))
                        target = target.view(-1, target.size(-1))

                        val_loss = loss_fn(projected_onehot_vectors, target, torch.ones(projected_onehot_vectors.size(0)).to(device))
                        total_val_loss_list[index] += val_loss.item()

                avg_val_loss_list = [total_loss / len(test_loader) for total_loss in total_val_loss_list]

                # 早期終了の判定とモデルの保存
                for index in range(len(extract_repr_layers)):
                    if is_early_stopped[index]:
                        continue

                    if avg_val_loss_list[index] < min_val_loss_list[index]:
                        min_val_loss_list[index] = avg_val_loss_list[index]
                        best_model_state_list[index] = linear_list[index].state_dict()
                        epochs_without_improvement_list[index] = 0
                    else:
                        epochs_without_improvement_list[index] += 1
                
                    if epochs_without_improvement_list[index] >= early_stopping_patience:
                        print(f"Layer {extract_repr_layers[index]} early stopped at epoch {epoch+1}")
                        is_early_stopped[index] = True
                    
                    metrics_history_list[index]["epoch"].append(epoch + 1)
                    metrics_history_list[index]["train_loss"].append(avg_loss_list[index])
                    metrics_history_list[index]["val_loss"].append(avg_val_loss_list[index])

                if all(is_early_stopped):
                    print(f"All layers early stopped at epoch {epoch+1}")
                    break
        
        for index in range(len(extract_repr_layers)):
            metrics_history_df = pd.DataFrame(metrics_history_list[index])
            metrics_history_df.to_csv(sub_output_dir / f"fold_{fold}" / f"layer_{extract_repr_layers[index]}" / "metrics_history.csv", index=False)
            print(f"Fold {fold} - Training history saved to {sub_output_dir / f'fold_{fold}' / f'layer_{extract_repr_layers[index]}' / 'metrics_history.csv'}")
            
            best_model_state = best_model_state_list[index]
            if best_model_state is not None:
                torch.save(best_model_state, sub_output_dir / f"fold_{fold}" / f"layer_{extract_repr_layers[index]}" / "best_model.pth")
                print(f"Fold {fold} - Best model saved to {sub_output_dir / f'fold_{fold}' / f'layer_{extract_repr_layers[index]}' / 'best_model.pth'}")
        
        # テストループ
        print("Testing...")
        
        result_dict_list = [{
            "seq_id": [],
            f"cosine_similarity_{extract_repr_layers[index]}": [],
        } for index in range(len(extract_repr_layers))]
        
        for linear in linear_list:
            linear.eval()
        
        with tqdm(total=len(test_loader), desc=f"Testing Fold {fold}") as test_pbar:
            for batch in test_loader:
                seq_ids = batch["seq_ids"]
                max_length = max(batch["lengths"])
                # onehotデータの取得とパディング, テンソル化
                onehot_vectors = []
                for seq_id in seq_ids:
                    onehot_vector = onehot_data[seq_id][:max_length]  # (L, 4)
                    if onehot_vector.shape[0] < max_length:
                        padding = np.zeros((max_length - onehot_vector.shape[0], 4))
                        onehot_vector = np.vstack([onehot_vector, padding])
                    onehot_vectors.append(onehot_vector)
                onehot_vectors = np.stack(onehot_vectors, axis=0)  # (B, L, 4)
                onehot_vectors = torch.tensor(onehot_vectors, dtype=torch.float32).to(device)  # (B, L, 4)
                
                with torch.no_grad():
                    targets = models[f"{to_framework}"]["model"]._test(batch, extract_repr_layers)["repr"]

                for index in range(len(extract_repr_layers)):
                    target = targets[index]
                    linear = linear_list[index]
                    
                    projected_onehot_vectors = linear(onehot_vectors)

                    projected_onehot_vectors = projected_onehot_vectors.view(-1, projected_onehot_vectors.size(-1)) # (B, L, E) -> (B*L, E)
                    target = target.view(-1, target.size(-1))     # (B, L, E) -> (B*L, E)

                    test_loss = loss_fn(projected_onehot_vectors, target, torch.ones(projected_onehot_vectors.size(0)).to(device))
                    cosine_sim = torch.nn.functional.cosine_similarity(projected_onehot_vectors, target, dim=-1)   # (B*L,)

                    cosine_sims = cosine_sim.view(len(seq_ids), -1).mean(dim=-1).detach().cpu().numpy().tolist() # (B, L) -> (B,)

                    # 結果をDataFrameに追加
                    result_dict_list[index]["seq_id"].extend(seq_ids)
                    result_dict_list[index][f"cosine_similarity_{extract_repr_layers[index]}"].extend(cosine_sims)

        df_list = []
        for index in range(len(extract_repr_layers)):
            df_list.append(pd.DataFrame(result_dict_list[index]))
        
        result_df = pd.concat(df_list, axis=1)
        result_df.to_csv(sub_output_dir / f"fold_{fold}" / f"results.csv", index=False)
        print(f"Fold {fold} - Test results saved to {sub_output_dir / f'fold_{fold}' / f'results.csv'}")

    all_folds_results = []
    for fold in range(num_folds):
        fold_result_df = pd.read_csv(sub_output_dir / f"fold_{fold}" / f"results.csv")
        fold_result_df["fold"] = fold
        all_folds_results.append(fold_result_df)
    final_result_df = pd.concat(all_folds_results, axis=0)
    final_result_df.to_csv(sub_output_dir / f"all_folds_results.csv", index=False)
    print(f"All folds results saved to {sub_output_dir / f'all_folds_results.csv'}")

Fold 0:
Training...


Fold 0:   1%|          | 95/9700 [00:17<29:29,  5.43it/s]


KeyboardInterrupt: 

## 実験結果や予測結果を可視化する

### CKAを使って定量化

In [ ]:
# ckaの計算
from torch_cka import CKA

test_loaders = {
    framework_name: create_dataloader(
        config=models[framework_name]["cfg"],
        split="test",
        dataset_dfs=archiveII_kfold_dfs[fold],
        use_additional_token=False,
    )
    for framework_name in frameworks if framework_name != "rinalmo"
}

cka = CKA(
    model1=models["mlm"]["model"],
    model2=models["data2vec"]["model"],
    model1_name="MLM",
    model2_name="data2vec",
    device=device,
)

cka.compare(
    dataloader1=test_loaders["mlm"],
    dataloader2=test_loaders["data2vec"],
)

results = cka.export()

cka.plot_results(results, output_path=output_dir / "cka_comparison.png")

| Comparing features |:   0%|          | 0/194 [00:00<?, ?it/s]

AttributeError: 'str' object has no attribute 'to'

### ロス推移の可視化

In [ ]:

for fold in range(num_folds):
    
    # mlm_to_data2vecのロス推移
    metrics_history_path = output_dir / f"mlm_to_data2vec" / f"fold_{fold}_metrics_history.csv"
    metrics_history_df = pd.read_csv(metrics_history_path)

    output_path = output_dir / f"mlm_to_data2vec" / f"fold_{fold}_loss_history.png"

    plt.figure(figsize=(10, 6))
    sns.lineplot(x="epoch", y="train_loss", data=metrics_history_df, label="Train Loss", color=framework_colors["mlm"])
    sns.lineplot(x="epoch", y="val_loss", data=metrics_history_df, label="Validation Loss", color=framework_colors["mlm"], linestyle="--")
    plt.ylim(0, None)
    plt.title(f"MLM to data2vec - Loss History")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.savefig(output_path)
    plt.close()

    # data2vec_to_mlmのロス推移
    metrics_history_path = output_dir / f"data2vec_to_mlm" / f"fold_{fold}_metrics_history.csv"
    metrics_history_df = pd.read_csv(metrics_history_path)
    output_path = output_dir / f"data2vec_to_mlm" / f"fold_{fold}_loss_history.png"
    plt.figure(figsize=(10, 6))
    sns.lineplot(x="epoch", y="train_loss", data=metrics_history_df, label="Train Loss", color=framework_colors["data2vec"])
    sns.lineplot(x="epoch", y="val_loss", data=metrics_history_df, label="Validation Loss", color=framework_colors["data2vec"], linestyle="--")
    plt.ylim(0, None)
    plt.title(f"data2vec to MLM - Loss History")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.savefig(output_path)
    plt.close()


#### 複数層の場合

In [9]:
# 層ごとに2つの実験のmetrics_historyを重ねて可視化

loss_output_dir_per_layer = output_dir / "per_layer_analysis" / "loss_history"
loss_output_dir_per_layer.mkdir(parents=True, exist_ok=True)

for fold in range(num_folds):
    for index in range(len(extract_repr_layers)):
        layer = extract_repr_layers[index]
        
        mlm_to_data2vec_path = output_dir / f"mlm_to_data2vec" / f"fold_{fold}" / f"layer_{layer}" / "metrics_history.csv"
        data2vec_to_mlm_path = output_dir / f"data2vec_to_mlm" / f"fold_{fold}" / f"layer_{layer}" / "metrics_history.csv"
        
        mlm_to_data2vec_df = pd.read_csv(mlm_to_data2vec_path)
        data2vec_to_mlm_df = pd.read_csv(data2vec_to_mlm_path)
        
        output_path = loss_output_dir_per_layer / f"fold_{fold}_layer_{layer}_loss_history.png"
        
        plt.figure(figsize=(10, 6))
        sns.lineplot(x="epoch", y="train_loss", data=mlm_to_data2vec_df, label="MLM to data2vec - Train Loss", color=framework_colors["mlm"])
        sns.lineplot(x="epoch", y="val_loss", data=mlm_to_data2vec_df, label="MLM to data2vec - Validation Loss", color=framework_colors["mlm"], linestyle="--")
        sns.lineplot(x="epoch", y="train_loss", data=data2vec_to_mlm_df, label="data2vec to MLM - Train Loss", color=framework_colors["data2vec"])
        sns.lineplot(x="epoch", y="val_loss", data=data2vec_to_mlm_df, label="data2vec to MLM - Validation Loss", color=framework_colors["data2vec"], linestyle="--")
        plt.ylim(0, None)
        plt.title(f"Fold {fold} - Layer {layer} - Loss History Comparison")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.legend()
        plt.savefig(output_path)
        plt.close()            
            

### テスト結果（cosine similarityの分布）の可視化

In [ ]:
# テスト結果の可視化(mlm_to_data2vecとdata2vec_to_mlmを比較)
all_results = {
    "mlm_to_data2vec": pd.DataFrame(),
    "data2vec_to_mlm": pd.DataFrame(),
    "onehot_to_mlm": pd.DataFrame(),
    "onehot_to_data2vec": pd.DataFrame(),
}

for fold in range(num_folds):
    mlm_to_data2vec_results_path = output_dir / f"mlm_to_data2vec" / f"fold_{fold}_results.csv"
    data2vec_to_mlm_results_path = output_dir / f"data2vec_to_mlm" / f"fold_{fold}_results.csv"
    onehot_to_mlm_results_path = output_dir / f"onehot_to_mlm" / f"fold_{fold}_results.csv"
    onehot_to_data2vec_results_path = output_dir / f"onehot_to_data2vec" / f"fold_{fold}_results.csv"

    mlm_to_data2vec_df = pd.read_csv(mlm_to_data2vec_results_path)
    data2vec_to_mlm_df = pd.read_csv(data2vec_to_mlm_results_path)
    onehot_to_mlm_df = pd.read_csv(onehot_to_mlm_results_path)
    onehot_to_data2vec_df = pd.read_csv(onehot_to_data2vec_results_path)

    all_results["mlm_to_data2vec"] = pd.concat([all_results["mlm_to_data2vec"], mlm_to_data2vec_df], ignore_index=True)
    all_results["data2vec_to_mlm"] = pd.concat([all_results["data2vec_to_mlm"], data2vec_to_mlm_df], ignore_index=True)
    all_results["onehot_to_mlm"] = pd.concat([all_results["onehot_to_mlm"], onehot_to_mlm_df], ignore_index=True)
    all_results["onehot_to_data2vec"] = pd.concat([all_results["onehot_to_data2vec"], onehot_to_data2vec_df], ignore_index=True)

plt.figure(figsize=(10, 6))
sns.kdeplot(all_results["mlm_to_data2vec"]["cosine_similarity"], label="MLM to data2vec", color=framework_colors["mlm"], fill=True, alpha=0.5)
sns.kdeplot(all_results["data2vec_to_mlm"]["cosine_similarity"], label="data2vec to MLM", color=framework_colors["data2vec"], fill=True, alpha=0.5)
sns.kdeplot(all_results["onehot_to_mlm"]["cosine_similarity"], label="One-Hot to MLM", color=framework_colors["onehot"], fill=True, alpha=0.5)
sns.kdeplot(all_results["onehot_to_data2vec"]["cosine_similarity"], label="One-Hot to data2vec", color=framework_colors["onehot"], fill=True, alpha=0.5)
plt.title("Cosine Similarity Distribution")
plt.xlabel("Cosine Similarity")
plt.ylabel("Density")
plt.legend()
plt.savefig(output_dir / "cosine_similarity_distribution.png")
plt.close()


#### 複数層の結果

In [11]:
all_results = {
    "mlm_to_data2vec": pd.read_csv(output_dir / f"mlm_to_data2vec" / f"all_folds_results.csv"),
    "data2vec_to_mlm": pd.read_csv(output_dir / f"data2vec_to_mlm" / f"all_folds_results.csv"),
    "onehot_to_mlm": pd.read_csv(output_dir / f"onehot_to_mlm" / f"all_folds_results.csv"),
    "onehot_to_data2vec": pd.read_csv(output_dir / f"onehot_to_data2vec" / f"all_folds_results.csv"),
}
output_dir_per_layer = output_dir / "per_layer_analysis"
output_dir_per_layer.mkdir(parents=True, exist_ok=True)

for layer in extract_repr_layers:
    mlm_to_data2vec_layer_col = f"cosine_similarity_{layer}"
    data2vec_to_mlm_layer_col = f"cosine_similarity_{layer}"
    onehot_to_mlm_layer_col = f"cosine_similarity_{layer}"
    onehot_to_data2vec_layer_col = f"cosine_similarity_{layer}"

    plt.figure(figsize=(10, 6))
    sns.kdeplot(all_results["mlm_to_data2vec"][mlm_to_data2vec_layer_col], label=f"MLM to data2vec - Layer {layer}", color=framework_colors["mlm"], fill=True, alpha=0.5)
    sns.kdeplot(all_results["data2vec_to_mlm"][data2vec_to_mlm_layer_col], label=f"data2vec to MLM - Layer {layer}", color=framework_colors["data2vec"], fill=True, alpha=0.5)
    sns.kdeplot(all_results["onehot_to_mlm"][onehot_to_mlm_layer_col], label=f"One-hot to MLM - Layer {layer}", color=framework_colors["onehot_to_mlm"], fill=True, alpha=0.5)
    sns.kdeplot(all_results["onehot_to_data2vec"][onehot_to_data2vec_layer_col], label=f"One-hot to data2vec - Layer {layer}", color=framework_colors["onehot_to_data2vec"], fill=True, alpha=0.5)
    plt.title(f"Cosine Similarity Distribution - Layer {layer}")
    plt.xlabel("Cosine Similarity")
    plt.ylabel("Density")
    plt.legend()
    plt.savefig(output_dir_per_layer / f"cosine_similarity_distribution_layer_{layer}.png")
    plt.close()

### テスト結果の定量的評価

In [26]:
# 定量化
mlm_to_data2vec_sims = all_results["mlm_to_data2vec"]["cosine_similarity"].values
data2vec_to_mlm_sims = all_results["data2vec_to_mlm"]["cosine_similarity"].values

t_stat, p_value = ttest_ind(mlm_to_data2vec_sims, data2vec_to_mlm_sims)
print(f"T-statistic: {t_stat:.4f}, P-value: {p_value:.4e}")

# 基本統計量の計算
mlm_to_data2vec_mean = np.mean(mlm_to_data2vec_sims)
mlm_to_data2vec_std = np.std(mlm_to_data2vec_sims)
data2vec_to_mlm_mean = np.mean(data2vec_to_mlm_sims)
data2vec_to_mlm_std = np.std(data2vec_to_mlm_sims)

print(f"MLM to data2vec - Mean: {mlm_to_data2vec_mean:.4f}, Std: {mlm_to_data2vec_std:.4f}")
print(f"data2vec to MLM - Mean: {data2vec_to_mlm_mean:.4f}, Std: {data2vec_to_mlm_std:.4f}")

# 結果の保存
quantitative_results = {
    "t_statistic": t_stat,
    "p_value": p_value,
    "mlm_to_data2vec_mean": mlm_to_data2vec_mean,
    "mlm_to_data2vec_std": mlm_to_data2vec_std,
    "data2vec_to_mlm_mean": data2vec_to_mlm_mean,
    "data2vec_to_mlm_std": data2vec_to_mlm_std,
}
quantitative_results_df = pd.DataFrame([quantitative_results])
quantitative_results_df.to_csv(output_dir / "quantitative_results.csv", index=False)
print(f"Quantitative results saved to {output_dir / 'quantitative_results.csv'}")

T-statistic: -19.2878, P-value: 5.3427e-81
MLM to data2vec - Mean: 0.8821, Std: 0.0626
data2vec to MLM - Mean: 0.9077, Std: 0.0537
Quantitative results saved to results/notebook/predict_another_repr/quantitative_results.csv


#### 複数層の場合

In [12]:
quantitative_results_dict = {
    f"layer_{layer}": {
        "t_statistic": None,
        "p_value": None,
        "mlm_to_data2vec_mean": None,
        "mlm_to_data2vec_std": None,
        "data2vec_to_mlm_mean": None,
        "data2vec_to_mlm_std": None,
    } for layer in extract_repr_layers
}

for layer in extract_repr_layers:
    mlm_to_data2vec_layer_col = f"cosine_similarity_{layer}"
    data2vec_to_mlm_layer_col = f"cosine_similarity_{layer}"
    onehot_to_mlm_layer_col = f"cosine_similarity_{layer}"
    onehot_to_data2vec_layer_col = f"cosine_similarity_{layer}"

    mlm_to_data2vec_sims = all_results["mlm_to_data2vec"][mlm_to_data2vec_layer_col].values
    data2vec_to_mlm_sims = all_results["data2vec_to_mlm"][data2vec_to_mlm_layer_col].values
    onehot_to_mlm_sims = all_results["onehot_to_mlm"][onehot_to_mlm_layer_col].values
    onehot_to_data2vec_sims = all_results["onehot_to_data2vec"][onehot_to_data2vec_layer_col].values

    t_stat, p_value = ttest_ind(mlm_to_data2vec_sims, data2vec_to_mlm_sims)

    mlm_to_data2vec_mean = np.mean(mlm_to_data2vec_sims)
    mlm_to_data2vec_std = np.std(mlm_to_data2vec_sims)
    data2vec_to_mlm_mean = np.mean(data2vec_to_mlm_sims)
    data2vec_to_mlm_std = np.std(data2vec_to_mlm_sims)
    onehot_to_mlm_mean = np.mean(onehot_to_mlm_sims)
    onehot_to_mlm_std = np.std(onehot_to_mlm_sims)
    onehot_to_data2vec_mean = np.mean(onehot_to_data2vec_sims)
    onehot_to_data2vec_std = np.std(onehot_to_data2vec_sims)

    quantitative_results_dict[f"layer_{layer}"] = {
        "t_statistic": t_stat,
        "p_value": p_value,
        "mlm_to_data2vec_mean": mlm_to_data2vec_mean,
        "mlm_to_data2vec_std": mlm_to_data2vec_std,
        "data2vec_to_mlm_mean": data2vec_to_mlm_mean,
        "data2vec_to_mlm_std": data2vec_to_mlm_std,
        "onehot_to_mlm_mean": onehot_to_mlm_mean,
        "onehot_to_mlm_std": onehot_to_mlm_std,
        "onehot_to_data2vec_mean": onehot_to_data2vec_mean,
        "onehot_to_data2vec_std": onehot_to_data2vec_std,
    }

quantitative_results_per_layer_df = pd.DataFrame.from_dict(quantitative_results_dict, orient="index")
quantitative_results_per_layer_df.to_csv(output_dir / "quantitative_results_per_layer.csv")
print(f"Quantitative results per layer saved to {output_dir / 'quantitative_results_per_layer.csv'}")

Quantitative results per layer saved to results/notebook/predict_another_repr/quantitative_results_per_layer.csv


### ファミリーに着目した分析

In [8]:
# 各ファミリーごとに結果を分析
family_results = {
    "mlm_to_data2vec": {},
    "data2vec_to_mlm": {},
}

for framework_name in ["mlm_to_data2vec", "data2vec_to_mlm"]:
    for family_name in family:
        family_df = all_results[framework_name][all_results[framework_name]["seq_id"].str.contains(family_name)]
        sims = family_df["cosine_similarity"].values
        mean_sim = np.mean(sims)
        std_sim = np.std(sims)
        family_results[framework_name][family_name] = {
            "mean": mean_sim,
            "std": std_sim,
            "count": len(sims),
        }
        print(f"{framework_name} - {family_name}: Mean={mean_sim:.4f}, Std={std_sim:.4f}, Count={len(sims)}")

# ファミリーごとの結果を可視化
family_names = list(family_results["mlm_to_data2vec"].keys())
mlm_means = [family_results["mlm_to_data2vec"][fam]["mean"] for fam in family_names]
mlm_stds = [family_results["mlm_to_data2vec"][fam]["std"] for fam in family_names]
data2vec_means = [family_results["data2vec_to_mlm"][fam]["mean"] for fam in family_names]
data2vec_stds = [family_results["data2vec_to_mlm"][fam]["std"] for fam in family_names]
x = np.arange(len(family_names))
width = 0.35
plt.figure(figsize=(12, 6))
plt.bar(x - width/2, mlm_means, width, yerr=mlm_stds, label="MLM to data2vec", color=framework_colors["mlm"])
plt.bar(x + width/2, data2vec_means, width, yerr=data2vec_stds, label="data2vec to MLM", color=framework_colors["data2vec"])
plt.xticks(x, family_names, rotation=45)
plt.title("Cosine Similarity by Family")
plt.xlabel("Family")
plt.ylabel("Mean Cosine Similarity")
plt.legend()
plt.tight_layout()
plt.savefig(output_dir / "cosine_similarity_by_family.png")
plt.close()

mlm_to_data2vec - 5s: Mean=0.9192, Std=0.0378, Count=1283
mlm_to_data2vec - 16s: Mean=0.6973, Std=0.1369, Count=66
mlm_to_data2vec - 23s: Mean=0.7544, Std=0.0346, Count=15
mlm_to_data2vec - grp1: Mean=0.8395, Std=0.0241, Count=74
mlm_to_data2vec - RNaseP: Mean=0.8233, Std=0.0299, Count=454
mlm_to_data2vec - srp: Mean=0.8829, Std=0.0581, Count=918
mlm_to_data2vec - telomerase: Mean=0.8813, Std=0.0104, Count=35
mlm_to_data2vec - tmRNA: Mean=0.8368, Std=0.0162, Count=462
mlm_to_data2vec - tRNA: Mean=0.9118, Std=0.0413, Count=557
data2vec_to_mlm - 5s: Mean=0.9367, Std=0.0304, Count=1283
data2vec_to_mlm - 16s: Mean=0.8158, Std=0.0782, Count=66
data2vec_to_mlm - 23s: Mean=0.8064, Std=0.0405, Count=15
data2vec_to_mlm - grp1: Mean=0.9440, Std=0.0202, Count=74
data2vec_to_mlm - RNaseP: Mean=0.8494, Std=0.0465, Count=454
data2vec_to_mlm - srp: Mean=0.9120, Std=0.0409, Count=918
data2vec_to_mlm - telomerase: Mean=0.9585, Std=0.0046, Count=35
data2vec_to_mlm - tmRNA: Mean=0.8455, Std=0.0360, Count

#### 複数層の場合

In [13]:
family_results_dict = {
    f"layer_{layer}": {
        "mlm_to_data2vec": {},
        "data2vec_to_mlm": {},
        "onehot_to_mlm": {},
        "onehot_to_data2vec": {},
    } for layer in extract_repr_layers
}

for layer in extract_repr_layers:
    mlm_to_data2vec_layer_col = f"cosine_similarity_{layer}"
    data2vec_to_mlm_layer_col = f"cosine_similarity_{layer}"
    onehot_to_mlm_layer_col = f"cosine_similarity_{layer}"
    onehot_to_data2vec_layer_col = f"cosine_similarity_{layer}"

    for family_name in family:
        mlm_family_df = all_results["mlm_to_data2vec"][all_results["mlm_to_data2vec"]["seq_id"].str.contains(family_name)]
        data2vec_family_df = all_results["data2vec_to_mlm"][all_results["data2vec_to_mlm"]["seq_id"].str.contains(family_name)]
        onehot_to_mlm_family_df = all_results["onehot_to_mlm"][all_results["onehot_to_mlm"]["seq_id"].str.contains(family_name)]
        onehot_to_data2vec_family_df = all_results["onehot_to_data2vec"][all_results["onehot_to_data2vec"]["seq_id"].str.contains(family_name)]

        mlm_sims = mlm_family_df[mlm_to_data2vec_layer_col].values
        data2vec_sims = data2vec_family_df[data2vec_to_mlm_layer_col].values
        onehot_to_mlm_sims = onehot_to_mlm_family_df[onehot_to_mlm_layer_col].values
        onehot_to_data2vec_sims = onehot_to_data2vec_family_df[onehot_to_data2vec_layer_col].values

        mlm_mean_sim = np.mean(mlm_sims)
        mlm_std_sim = np.std(mlm_sims)
        data2vec_mean_sim = np.mean(data2vec_sims)
        data2vec_std_sim = np.std(data2vec_sims)
        onehot_to_mlm_mean_sim = np.mean(onehot_to_mlm_sims)
        onehot_to_mlm_std_sim = np.std(onehot_to_mlm_sims)
        onehot_to_data2vec_mean_sim = np.mean(onehot_to_data2vec_sims)
        onehot_to_data2vec_std_sim = np.std(onehot_to_data2vec_sims)

        family_results_dict[f"layer_{layer}"]["mlm_to_data2vec"][family_name] = {
            "mean": mlm_mean_sim,
            "std": mlm_std_sim,
            "count": len(mlm_sims),
        }
        family_results_dict[f"layer_{layer}"]["data2vec_to_mlm"][family_name] = {
            "mean": data2vec_mean_sim,
            "std": data2vec_std_sim,
            "count": len(data2vec_sims),
        }
        family_results_dict[f"layer_{layer}"]["onehot_to_mlm"][family_name] = {
            "mean": onehot_to_mlm_mean_sim,
            "std": onehot_to_mlm_std_sim,
            "count": len(onehot_to_mlm_sims),
        }
        family_results_dict[f"layer_{layer}"]["onehot_to_data2vec"][family_name] = {
            "mean": onehot_to_data2vec_mean_sim,
            "std": onehot_to_data2vec_std_sim,
            "count": len(onehot_to_data2vec_sims),
        }
        print(f"Layer {layer} - {family_name}: MLM to data2vec Mean={mlm_mean_sim:.4f}, Std={mlm_std_sim:.4f}, Count={len(mlm_sims)} | data2vec to MLM Mean={data2vec_mean_sim:.4f}, Std={data2vec_std_sim:.4f}, Count={len(data2vec_sims)} | One-hot to MLM Mean={onehot_to_mlm_mean_sim:.4f}, Std={onehot_to_mlm_std_sim:.4f}, Count={len(onehot_to_mlm_sims)} | One-hot to data2vec Mean={onehot_to_data2vec_mean_sim:.4f}, Std={onehot_to_data2vec_std_sim:.4f}, Count={len(onehot_to_data2vec_sims)}")

# ファミリーごとの結果をレイヤーごとに可視化
for layer in extract_repr_layers:
    family_names = list(family_results_dict[f"layer_{layer}"]["mlm_to_data2vec"].keys())
    mlm_means = [family_results_dict[f"layer_{layer}"]["mlm_to_data2vec"][fam]["mean"] for fam in family_names]
    mlm_stds = [family_results_dict[f"layer_{layer}"]["mlm_to_data2vec"][fam]["std"] for fam in family_names]
    data2vec_means = [family_results_dict[f"layer_{layer}"]["data2vec_to_mlm"][fam]["mean"] for fam in family_names]
    data2vec_stds = [family_results_dict[f"layer_{layer}"]["data2vec_to_mlm"][fam]["std"] for fam in family_names]
    onehot_to_mlm_means = [family_results_dict[f"layer_{layer}"]["onehot_to_mlm"][fam]["mean"] for fam in family_names]
    onehot_to_mlm_stds = [family_results_dict[f"layer_{layer}"]["onehot_to_mlm"][fam]["std"] for fam in family_names]
    onehot_to_data2vec_means = [family_results_dict[f"layer_{layer}"]["onehot_to_data2vec"][fam]["mean"] for fam in family_names]
    onehot_to_data2vec_stds = [family_results_dict[f"layer_{layer}"]["onehot_to_data2vec"][fam]["std"] for fam in family_names]
    x = np.arange(len(family_names))
    width = 0.35
    plt.figure(figsize=(12, 6))
    plt.bar(x - width/2, mlm_means, width, yerr=mlm_stds, label="MLM to data2vec", color=framework_colors["mlm"])
    plt.bar(x + width/2, data2vec_means, width, yerr=data2vec_stds, label="data2vec to MLM", color=framework_colors["data2vec"])
    plt.bar(x, onehot_to_mlm_means, width, yerr=onehot_to_mlm_stds, label="One-hot to MLM", color=framework_colors["onehot_to_mlm"])
    plt.bar(x + width, onehot_to_data2vec_means, width, yerr=onehot_to_data2vec_stds, label="One-hot to data2vec", color=framework_colors["onehot_to_data2vec"])
    plt.xticks(x, family_names, rotation=45)
    plt.title(f"Cosine Similarity by Family - Layer {layer}")
    plt.xlabel("Family")
    plt.ylabel("Mean Cosine Similarity")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_dir_per_layer / f"cosine_similarity_by_family_layer_{layer}.png")
    plt.close()

Layer 0 - 5s: MLM to data2vec Mean=1.0000, Std=0.0000, Count=1283 | data2vec to MLM Mean=1.0000, Std=0.0000, Count=1283 | One-hot to MLM Mean=0.8935, Std=0.0268, Count=1283 | One-hot to data2vec Mean=0.9352, Std=0.0101, Count=1283
Layer 0 - 16s: MLM to data2vec Mean=1.0000, Std=0.0000, Count=66 | data2vec to MLM Mean=1.0000, Std=0.0000, Count=66 | One-hot to MLM Mean=0.9202, Std=0.0233, Count=66 | One-hot to data2vec Mean=0.8948, Std=0.0281, Count=66
Layer 0 - 23s: MLM to data2vec Mean=1.0000, Std=0.0000, Count=15 | data2vec to MLM Mean=1.0000, Std=0.0000, Count=15 | One-hot to MLM Mean=0.9189, Std=0.0142, Count=15 | One-hot to data2vec Mean=0.8866, Std=0.0133, Count=15
Layer 0 - grp1: MLM to data2vec Mean=1.0000, Std=0.0000, Count=74 | data2vec to MLM Mean=1.0000, Std=0.0000, Count=74 | One-hot to MLM Mean=0.9258, Std=0.0150, Count=74 | One-hot to data2vec Mean=0.8840, Std=0.0192, Count=74
Layer 0 - RNaseP: MLM to data2vec Mean=1.0000, Std=0.0000, Count=454 | data2vec to MLM Mean=1.00

onehot to {framework} の場合の層ごとの平均類似度推移を可視化

In [18]:
# onehot to {framework} の場合の層ごとの平均類似度推移を可視化
quantitative_results_per_layer_df = pd.read_csv(output_dir / "quantitative_results_per_layer.csv")
quantitative_results_per_layer_df.columns = ["layer"] + quantitative_results_per_layer_df.columns.tolist()[1:]
print(quantitative_results_per_layer_df)

plt.figure(figsize=(10, 6))
for framework_name in ["onehot_to_mlm", "onehot_to_data2vec"]:
    means = []
    stdevs = []
    for layer in extract_repr_layers:
        mean_col = f"{framework_name}_mean"
        stdev_col = f"{framework_name}_std"
        mean_value = quantitative_results_per_layer_df[quantitative_results_per_layer_df["layer"] == f"layer_{layer}"][mean_col].values[0]
        stdev_value = quantitative_results_per_layer_df[quantitative_results_per_layer_df["layer"] == f"layer_{layer}"][stdev_col].values[0]
        means.append(mean_value)
        stdevs.append(stdev_value)
    plt.errorbar(extract_repr_layers, means, yerr=stdevs, label=framework_name.replace("_", " "), marker=".", capsize=3, color=framework_colors[framework_name])
plt.title("Cosine Similarity by Layer - One-hot to Framework")
plt.xlabel("Layer")
plt.xticks(extract_repr_layers)
plt.ylabel("Mean Cosine Similarity")
plt.legend()
plt.savefig(output_dir / "onehot_to_framework_cosine_similarity_by_layer.png")
plt.close()

       layer  t_statistic        p_value  mlm_to_data2vec_mean  \
0    layer_0    48.144562   0.000000e+00              1.000000   
1    layer_1  -115.579127   0.000000e+00              0.971910   
2    layer_2   -90.265364   0.000000e+00              0.945092   
3    layer_3   -43.525280   0.000000e+00              0.950941   
4    layer_4    14.143099   7.450359e-45              0.963899   
5    layer_5    26.341386  1.636556e-146              0.958742   
6    layer_6    34.315671  2.514027e-240              0.951004   
7    layer_7    44.557386   0.000000e+00              0.943659   
8    layer_8    54.791101   0.000000e+00              0.939284   
9    layer_9    61.399852   0.000000e+00              0.936554   
10  layer_10    56.215207   0.000000e+00              0.919716   
11  layer_11    91.643874   0.000000e+00              0.966807   
12  layer_12   -12.978540   4.046162e-38              0.902493   

    mlm_to_data2vec_std  data2vec_to_mlm_mean  data2vec_to_mlm_std  \
0    